# DJ Set Rows Analysis

Analyze `dj_set_rows` to explore row signatures, class distributions, and sample rows.

In [1]:
# ============================================
# End-to-end pipeline (copy/paste)
# 1) Load rows from SQLite
# 2) Extract container <div ...> attributes (data-* keys via stored JSON, optional classes/flags)
# 3) Normalize away junk dimensions (DROP/QUOTIENT meaningless variability)
# 4) Build sparse binary feature matrix X (rows x features)
# 5) Exact schema clustering: group by identical feature-index keyset
# 6) Print cluster summaries + example URLs
# ============================================

import json
import re
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
from collections import Counter, defaultdict

from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize


# ------------------------
# CONFIG
# ------------------------
DB_PATH = Path("/Users/johncabrahams/Desktop/Projects/tracklist_engine/data/db/music_database.db")
LIMIT = 0  # 0 == no limit

# feature settings
MIN_FEATURE_DF = 5       # drop rare features (appearing in < MIN_FEATURE_DF rows)
MAX_FEATURES = 50000     # safety cap

INCLUDE_DATA_KEYS = True
INCLUDE_CLASS_TOKENS = True
INCLUDE_FLAGS = True

# Optional row normalization (not required for exact keyset grouping; keep False unless you need it elsewhere)
ROW_L2_NORMALIZE = False


# # -----------------------------
# # Step 2 — Normalize away junk dimensions (THIS is the fix)
# # -----------------------------

# # Always normalize these class patterns (numeric suffixes are not schema)
# CLASS_NORMALIZE_PATTERNS = [
#     (re.compile(r"^trRow\d+$"), "trRow#"),
#     (re.compile(r"^subPos\d+$"), "subPos#"),
#     (re.compile(r"^tlp_\d+$"), "tlp_#"),
# ]

# def normalize_class_token(tok: str) -> str:
#     for rx, repl in CLASS_NORMALIZE_PATTERNS:
#         if rx.match(tok):
#             return repl
#     return tok

# # Often normalize / drop these data keys (index-like, not schema)
# # (You can expand this list as you discover more index-like keys)
# DROP_DATA_KEYS = {
#     "data-pos",
#     "data-mashpos",
#     "data-subpos",
# }

# # Flags that are basically "id exists" noise (drop them entirely)
# DROP_FEATURES_EXACT = {
#     "flag:id_starts_tlp_",
#     "flag:div_has_id_attr",
# }

# def normalize_feature(feat: str) -> Optional[str]:
#     """
#     Global final pass: drop features that poison schema clustering.
#     Return None to DROP. Return string to keep (possibly rewritten later if desired).
#     """
#     if feat in DROP_FEATURES_EXACT:
#         return None
#     return feat


# # -----------------------------
# # Helpers: parse "container div" attributes
# # -----------------------------
# DIV_OPEN_RE = re.compile(r"<div\b([^>]*)>", re.IGNORECASE | re.DOTALL)

# ATTR_RE = re.compile(
#     r"""([a-zA-Z_:][-a-zA-Z0-9_:.]*)\s*=\s*(".*?"|'.*?'|[^\s>]+)""",
#     re.DOTALL
# )

# def _strip_quotes(v: str) -> str:
#     v = (v or "").strip()
#     if len(v) >= 2 and ((v[0] == v[-1] == '"') or (v[0] == v[-1] == "'")):
#         return v[1:-1]
#     return v

# def parse_first_div_attrs(raw_html: str) -> Dict[str, str]:
#     """
#     Extract attributes from the FIRST <div ...> opening tag.
#     This approximates 'container div' in your scraped row_html.
#     """
#     if not raw_html:
#         return {}
#     m = DIV_OPEN_RE.search(raw_html)
#     if not m:
#         return {}
#     attrs_blob = m.group(1)
#     out = {}
#     for am in ATTR_RE.finditer(attrs_blob):
#         k = am.group(1).strip().lower()
#         v = _strip_quotes(am.group(2))
#         out[k] = v
#     return out

# def class_tokens_from_attrs(attrs: Dict[str, str]) -> List[str]:
#     cls = attrs.get("class", "") or ""
#     toks = [t for t in cls.split() if t]
#     return [normalize_class_token(t) for t in toks]

# def has_style_display_none(attrs: Dict[str, str], raw_html: str) -> bool:
#     style = (attrs.get("style") or "").lower()
#     if "display" in style and "none" in style:
#         return True
#     return "display: none" in (raw_html or "").lower()

# def safe_json_keys(data_attrs_json: Optional[str]) -> List[str]:
#     """
#     Your stored JSON of data-attrs may be the most reliable source
#     of data-* keys at the container level.
#     """
#     if not data_attrs_json:
#         return []
#     try:
#         d = json.loads(data_attrs_json)
#         if isinstance(d, dict):
#             return [str(k) for k in d.keys()]
#     except Exception:
#         return []
#     return []


# -----------------------------
# Load DB rows
# -----------------------------
@dataclass
class Row:
    set_id: str
    set_url: str
    element_id: str
    classes: str
    data_attrs_json: str
    text_excerpt: str
    raw_html: str

def load_rows(db_path: Path, limit: int = 0) -> List[Row]:
    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()
    q = """
    SELECT r.set_id, s.set_url, r.element_id, r.classes, r.data_attrs_json, r.text_excerpt, r.raw_html
    FROM dj_set_rows r
    LEFT JOIN dj_sets s ON s.set_id = r.set_id
    """
    if limit and limit > 0:
        q += f" LIMIT {int(limit)}"
    cur.execute(q)

    rows: List[Row] = []
    for set_id, set_url, element_id, classes, data_attrs_json, text_excerpt, raw_html in cur.fetchall():
        rows.append(Row(
            set_id=str(set_id) if set_id is not None else "",
            set_url=str(set_url) if set_url is not None else "",
            element_id=str(element_id) if element_id is not None else "",
            classes=str(classes) if classes is not None else "",
            data_attrs_json=str(data_attrs_json) if data_attrs_json is not None else "",
            text_excerpt=str(text_excerpt) if text_excerpt is not None else "",
            raw_html=str(raw_html) if raw_html is not None else "",
        ))
    conn.close()
    return rows


# # -----------------------------
# # Feature extraction (with junk normalization)
# # -----------------------------
# def extract_features(row: Row) -> List[str]:
#     feats: List[str] = []

#     attrs = parse_first_div_attrs(row.raw_html)
#     cls_tokens = class_tokens_from_attrs(attrs)

#     # (A) data-* presence on container (prefer stored JSON keys)
#     data_keys = safe_json_keys(row.data_attrs_json)

#     if INCLUDE_DATA_KEYS:
#         for k in data_keys:
#             k_norm = str(k).strip()
#             # Drop index-like keys (pos/subpos/mashpos etc.)
#             if k_norm in DROP_DATA_KEYS:
#                 continue
#             feats.append(f"data:{k_norm}")

#     # (B) class tokens on container div (already normalized)
#     if INCLUDE_CLASS_TOKENS:
#         for t in cls_tokens:
#             feats.append(f"class:{t}")

#     # (C) structural flags (normalize away the id-noise later via normalize_feature)
#     if INCLUDE_FLAGS:
#         if "onclick" in attrs:
#             feats.append("flag:has_onclick")
#         if "href" in (row.raw_html or "").lower():
#             feats.append("flag:has_href_anywhere")
#         if row.element_id:
#             feats.append("flag:has_element_id")

#         if attrs.get("id"):
#             feats.append("flag:div_has_id_attr")
#             if attrs.get("id", "").startswith("tlp_"):
#                 feats.append("flag:id_starts_tlp_")

#         if has_style_display_none(attrs, row.raw_html):
#             feats.append("flag:style_display_none")

#         if (row.text_excerpt or "").strip():
#             feats.append("text:nonempty")
#         else:
#             feats.append("text:empty")

#     # Final normalization / dropping pass (THIS is what you asked about)
#     normalized: List[str] = []
#     for f in feats:
#         nf = normalize_feature(f)
#         if nf is not None:
#             normalized.append(nf)

#     # If literally nothing extracted, keep a marker so rows aren't all-zero.
#     if not normalized:
#         normalized.append("feat:(none)")

#     # De-dupe within row
#     return sorted(set(normalized))


# # -----------------------------
# # Build sparse matrix + metadata
# # -----------------------------
# def build_sparse_matrix(rows: List[Row]) -> Tuple[csr_matrix, List[str], List[Dict[str, str]], List[List[str]]]:
#     """
#     Returns:
#       X (csr_matrix): n x d binary
#       feat_names (list[str]): column vocabulary
#       row_meta (list[dict]): urls/ids for debugging
#       row_feats (list[list[str]]): per-row feature lists (filtered to feat_names)
#     """
#     # Pass 1: compute DF counts
#     df: Dict[str, int] = {}
#     raw_row_feats: List[List[str]] = []
#     row_meta: List[Dict[str, str]] = []

#     for r in rows:
#         feats = extract_features(r)
#         raw_row_feats.append(feats)
#         for f in feats:
#             df[f] = df.get(f, 0) + 1

#         url = ""
#         if r.set_url and r.element_id:
#             url = f"{r.set_url}#{r.element_id}"

#         row_meta.append({
#             "set_id": r.set_id,
#             "element_id": r.element_id,
#             "url": url
#         })

#     # Filter by DF
#     kept = [f for f, c in df.items() if c >= MIN_FEATURE_DF]
#     kept.sort()
#     if len(kept) > MAX_FEATURES:
#         kept = kept[:MAX_FEATURES]

#     feat_index = {f: i for i, f in enumerate(kept)}

#     # Pass 2: CSR construction
#     indptr = [0]
#     indices: List[int] = []
#     data: List[float] = []
#     filtered_row_feats: List[List[str]] = []

#     for feats in raw_row_feats:
#         cols = sorted({feat_index[f] for f in feats if f in feat_index})
#         indices.extend(cols)
#         data.extend([1.0] * len(cols))
#         indptr.append(len(indices))
#         filtered_row_feats.append([kept[j] for j in cols])

#     X = csr_matrix((data, indices, indptr), shape=(len(rows), len(kept)), dtype=np.float32)
#     return X, kept, row_meta, filtered_row_feats


# # -----------------------------
# # Exact schema clustering (discrete)
# # -----------------------------
# def exact_schema_clusters(X: csr_matrix) -> Dict[Tuple[int, ...], List[int]]:
#     """
#     Group row indices by identical nonzero-column keyset.
#     """
#     groups: Dict[Tuple[int, ...], List[int]] = defaultdict(list)
#     for i in range(X.shape[0]):
#         k = tuple(X[i].indices)  # already sorted
#         groups[k].append(i)
#     return groups


# # -----------------------------
# # MAIN
# # -----------------------------
# rows = load_rows(DB_PATH, LIMIT)
# print(f"Rows loaded: {len(rows)}")

# X, feat_names, row_meta, row_feats = build_sparse_matrix(rows)
# print(f"Sparse X: shape={X.shape}, nnz={X.nnz}, density={X.nnz/(X.shape[0]*X.shape[1]+1e-12):.6f}")

# # Optional (not needed for exact keyset grouping)
# if ROW_L2_NORMALIZE:
#     X_work = normalize(X, norm="l2", axis=1, copy=True)
# else:
#     X_work = X

# groups = exact_schema_clusters(X_work)
# print(f"Exact schema clusters: {len(groups)}")

# # Print clusters by size + show their feature names
# schema_clusters = sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True)

# for cid, (k, members) in enumerate(schema_clusters):
#     feats = [feat_names[j] for j in k]
#     print(f"\n--- schema {cid} | size={len(members)} ---")
#     print("features:", feats)

#     # show up to 5 example urls
#     shown = 0
#     for i in members:
#         url = row_meta[i]["url"]
#         if url:
#             print("  ex:", url)
#             shown += 1
#         if shown >= 5:
#             break

# # Optional: define cluster_ids (schema id per row)
# cluster_ids = np.empty(X_work.shape[0], dtype=int)
# for cid, (k, members) in enumerate(schema_clusters):
#     for i in members:
#         cluster_ids[i] = cid


In [2]:
# Count DJ sets in DB (and html files if you want)
import sqlite3
from pathlib import Path

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM dj_sets")
print("dj_sets_count:", cur.fetchone()[0])
conn.close()



dj_sets_count: 8139


In [7]:
# from datetime import datetime, timezone
# from pathlib import Path
# from typing import Optional, Tuple, Dict, List
# import re
# import sqlite3

# DB_PATH = Path("/Users/johncabrahams/Desktop/Projects/tracklist_engine/web_crawler/data/db/music_database.db")
# # LIMIT = 0  # 0 == no limit

# # Captcha scan settings
# HTML_DIR = Path("/Users/johncabrahams/Desktop/Projects/tracklist_engine/web_crawler/data/html")
# CAPTCHA_MARKERS = [
#     "captcha",
#     "g-recaptcha",
#     "recaptcha",
#     "hcaptcha",
#     "cf-turnstile",
#     "cloudflare",
#     "cf-challenge",
#     "challenge-platform",
#     "attention required",
#     "verify you are human",
#     "perimeterx",
#     "distil",
#     "datadome",
#     "akamai",
# ]


# # -----------------------------
# # Captcha helpers
# # -----------------------------
# def _parse_timestamp_from_filename(name: str) -> Optional[datetime]:
#     """
#     Expected filename format: {set_id}_YYYYMMDDTHHMMSSZ.html
#     Example: 24fsn51k_20260131T014436Z.html
#     """
#     m = re.search(r"_([0-9]{8}T[0-9]{6}Z)\.html$", name)
#     if not m:
#         return None
#     try:
#         return datetime.strptime(m.group(1), "%Y%m%dT%H%M%SZ").replace(tzinfo=timezone.utc)
#     except ValueError:
#         return None

# def _find_most_recent_html(html_dir: Path) -> Optional[Path]:
#     html_files = list(html_dir.glob("*.html"))
#     if not html_files:
#         return None

#     def sort_key(p: Path):
#         ts = _parse_timestamp_from_filename(p.name)
#         if ts:
#             return ts
#         return datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc)

#     return max(html_files, key=sort_key)

# def _is_captcha_html(path: Path) -> Tuple[bool, List[str]]:
#     try:
#         content = path.read_text(errors="ignore").lower()
#     except Exception:
#         return False, []
#     hits = [m for m in CAPTCHA_MARKERS if m in content]
#     return (len(hits) > 0), hits

# def _load_set_urls(db_path: Path, set_ids: List[str]) -> Dict[str, str]:
#     if not set_ids:
#         return {}
#     conn = sqlite3.connect(str(db_path))
#     cur = conn.cursor()
#     q = f"SELECT set_id, set_url FROM dj_sets WHERE set_id IN ({','.join('?' for _ in set_ids)})"
#     cur.execute(q, set_ids)
#     mapping = {str(sid): str(url) for sid, url in cur.fetchall() if sid is not None and url is not None}
#     conn.close()
#     return mapping

# def _scan_html_dir_for_captchas(html_dir: Path, db_path: Path) -> List[Dict[str, str]]:
#     rows = []
#     html_files = list(html_dir.glob("*.html"))
#     if not html_files:
#         return rows

#     set_ids = []
#     for p in html_files:
#         set_id = p.name.split("_", 1)[0]
#         if set_id:
#             set_ids.append(set_id)

#     set_url_map = _load_set_urls(db_path, sorted(set(set_ids)))

#     for p in html_files:
#         is_captcha, hits = _is_captcha_html(p)
#         if not is_captcha:
#             continue
#         ts = _parse_timestamp_from_filename(p.name)
#         if ts is None:
#             ts = datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc)
#         set_id = p.name.split("_", 1)[0]
#         rows.append({
#             "set_id": set_id,
#             "set_url": set_url_map.get(set_id, ""),
#             "timestamp": ts.isoformat(),
#             "file": p.name,
#             "hits": ",".join(hits),
#         })

#     rows.sort(key=lambda r: r["timestamp"], reverse=True)
#     return rows


# # ------------------------
# # CAPTCHA SCAN (HTML folder)
# # ------------------------
# latest_html = _find_most_recent_html(HTML_DIR)
# if latest_html is None:
#     print(f"\nNo HTML files found in {HTML_DIR}")
# else:
#     latest_ts = _parse_timestamp_from_filename(latest_html.name)
#     if latest_ts is None:
#         latest_ts = datetime.fromtimestamp(latest_html.stat().st_mtime, tz=timezone.utc)
#     latest_is_captcha, latest_hits = _is_captcha_html(latest_html)
#     print("\n=== Latest HTML file ===")
#     print(f"file={latest_html.name} | ts={latest_ts.isoformat()} | captcha={latest_is_captcha} | hits={','.join(latest_hits)}")

# captcha_rows = _scan_html_dir_for_captchas(HTML_DIR, DB_PATH)
# print(f"\n=== Captcha pages in {HTML_DIR} ===")
# print(f"count={len(captcha_rows)}")

# all_html_files = list(HTML_DIR.glob("*.html"))
# if all_html_files:
#     captcha_pct = (len(captcha_rows) / len(all_html_files)) * 100.0
#     print(f"captcha_percent={captcha_pct:.2f}% ({len(captcha_rows)}/{len(all_html_files)})")
# else:
#     print("captcha_percent=n/a (no html files)")
# for row in captcha_rows:
#     print(f"{row['set_id']} | {row['timestamp']} | {row['file']} | url={row['set_url']} | hits={row['hits']}")



In [8]:
# # Delete captcha sets from all relevant tables
# import sqlite3

# captcha_set_ids = sorted({row['set_id'] for row in captcha_rows})
# print(f'captcha set_ids: {len(captcha_set_ids)}')

# if captcha_set_ids:
#     conn = sqlite3.connect(str(DB_PATH))
#     cur = conn.cursor()
#     try:
#         # Scrape failures are not FK-cascaded
#         cur.executemany("DELETE FROM scrape_failures WHERE set_id = ?", [(sid,) for sid in captcha_set_ids])
#         # Cascades will clear related tables, but delete explicitly to be safe/orderly
#         cur.executemany("DELETE FROM dj_set_track_media_links WHERE set_id = ?", [(sid,) for sid in captcha_set_ids])
#         cur.executemany("DELETE FROM dj_set_media_links WHERE set_id = ?", [(sid,) for sid in captcha_set_ids])
#         cur.executemany("DELETE FROM dj_set_rows WHERE set_id = ?", [(sid,) for sid in captcha_set_ids])
#         cur.executemany("DELETE FROM dj_set_crawls WHERE set_id = ?", [(sid,) for sid in captcha_set_ids])
#         cur.executemany("DELETE FROM dj_sets WHERE set_id = ?", [(sid,) for sid in captcha_set_ids])
#         conn.commit()
#         print('Deleted captcha sets from DB.')
#     finally:
#         conn.close()
# else:
#     print('No captcha set_ids found; nothing to delete.')


In [9]:
# # Delete captcha HTML files from disk
# from pathlib import Path

# captcha_files = []
# for row in captcha_rows:
#     fname = row.get('file')
#     if fname:
#         captcha_files.append(HTML_DIR / fname)

# deleted = 0
# missing = 0
# for path in captcha_files:
#     if path.exists():
#         path.unlink()
#         deleted += 1
#     else:
#         missing += 1

# print(f'deleted_html={deleted} missing_html={missing}')


In [10]:
# # Determine all <div> id values present in dj_set_rows
# import re
# from collections import Counter
# from pathlib import Path

# DIV_ID_RE = re.compile(r'<div\b[^>]*\bid\s*=\s*(".*?"|\'.*?\'|[^\s>]+)', re.IGNORECASE | re.DOTALL)

# HTML_DIR = Path("/Users/johncabrahams/Desktop/Projects/tracklist_engine/web_crawler/data/html")

# def _strip_quotes(v: str) -> str:
#     v = (v or '').strip()
#     if len(v) >= 2 and ((v[0] == v[-1] == '"') or (v[0] == v[-1] == "'")):
#         return v[1:-1]
#     return v

# # Use already-loaded rows if present; otherwise load from DB
# try:
#     _rows = rows
# except NameError:
#     _rows = load_rows(DB_PATH, LIMIT)

# # Map set_id -> example html file (most recent by timestamp in filename)
# _set_to_html = {}
# if HTML_DIR.exists():
#     for p in HTML_DIR.glob("*.html"):
#         name = p.name
#         if "_" not in name:
#             continue
#         set_id, _rest = name.split("_", 1)
#         if not set_id:
#             continue
#         prev = _set_to_html.get(set_id)
#         if prev is None or name > Path(prev).name:
#             _set_to_html[set_id] = str(p)

# # Collect ids + example file where seen
# id_to_example_file = {}
# div_ids = []
# for r in _rows:
#     example_file = _set_to_html.get(r.set_id, "")

#     # element_id column
#     if r.element_id:
#         div_ids.append(r.element_id)
#         id_to_example_file.setdefault(r.element_id, example_file)

#     # any id= on div tags inside raw_html
#     for m in DIV_ID_RE.finditer(r.raw_html or ''):
#         vid = _strip_quotes(m.group(1))
#         if vid:
#             div_ids.append(vid)
#             id_to_example_file.setdefault(vid, example_file)

# div_ids = [d for d in div_ids if d]
# unique_ids = sorted(set(div_ids))
# print(f'Unique div ids: {len(unique_ids)} (total occurrences: {len(div_ids)})')

# # Show common prefixes (e.g., tlp_, sug_) with one example html file

# def prefix_of(s: str) -> str:
#     for i, ch in enumerate(s):
#         if ch.isdigit():
#             return s[:i]
#     return s

# prefix_counts = Counter(prefix_of(s) for s in div_ids)

# # Pick one example file per prefix
# prefix_to_example_file = {}
# for vid in unique_ids:
#     pfx = prefix_of(vid)
#     if pfx not in prefix_to_example_file:
#         prefix_to_example_file[pfx] = id_to_example_file.get(vid, "")

# print('Top id prefixes (with example html file):')
# for pfx, cnt in prefix_counts.most_common(20):
#     print(f"  {pfx!r}: {cnt}	{prefix_to_example_file.get(pfx, '')}")


In [11]:
from bs4 import BeautifulSoup, Tag

# rows = load_rows(DB_PATH, LIMIT)

# # def process_html_row(row: str):
# row_soup = BeautifulSoup(rows[3].raw_html, 'html.parser')

In [12]:
from dataclasses import dataclass
from enum import StrEnum
from typing import FrozenSet


class TrackTokenType(StrEnum):
    REGULAR_TRACK = "regular_track"
    MASHUP_TRACK = "mashup_track"
    MASHUP_CONSTITUENT_TRACK = "mashup_constituent_track"
    ID_TRACK = "id_track"
    ID_REMIX_TRACK = "id_remix_track"


@dataclass(frozen=True, slots=True)
class TrackDOMAttrs:
    div_id: str | None
    data_id: str | None
    data_isid: str | None
    data_isided: str | None
    data_mashpos: str | None
    data_mashup: str | None
    data_noguestsug: str | None
    data_protected: str | None
    data_rbcst: str | None
    data_subpos: str | None
    data_trackid: str | None
    data_trno: str | None
    data_videos: str | None


@dataclass(frozen=True, slots=True)
class PositionalInformation:
    index: str | None
    time: str | None


@dataclass(frozen=True, slots=True)
class TrackMetadata:
    name: str | None
    by_artist: str | None
    publisher: str | None
    duration_seconds: int | None
    genre: str | None
    url: str | None


@dataclass(frozen=True, slots=True)
class ReliabilityMetrics:
    track_IDer: str | None
    total_tracklist_plays: str | None
    is_id_remix: bool


@dataclass(frozen=True, slots=True)
class TrackToken:
    token_types: FrozenSet[TrackTokenType]
    attrs: TrackDOMAttrs
    positional: PositionalInformation
    metadata: TrackMetadata
    reliability: ReliabilityMetrics


In [13]:
def _is_true(v) -> bool:
    if v is None:
        return False
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() not in {"", "0", "false", "none", "null"}

def parse_iso_duration(d: str | None):
    if not d:
        return None
    match = re.fullmatch(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", d.strip())
    if not match:
        return None

    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)

    return hours * 3600 + minutes * 60 + seconds


def get_positional_information(outer_div: Tag):
    # This is important positional information
    positional_information_div = outer_div.find("div", class_="bPlay")
    index = positional_information_div.find("span").text
    time = positional_information_div.find("div").text

    return index, time

def get_metadata(outer_div: Tag):
    name = outer_div.find("meta", itemprop="name")
    by_artist = outer_div.find("meta", itemprop="byArtist")
    publisher = outer_div.find("meta", itemprop="publisher")
    duration_tag = outer_div.find("meta", itemprop="duration")
    genre = outer_div.find("meta", itemprop="genre")
    url = outer_div.find("meta", itemprop="url")

    duration = None
    if duration_tag:
        duration_str = duration_tag.get("content") or duration_tag.get_text(strip=True)
        duration = parse_iso_duration(duration_str)

    return name, by_artist, publisher, duration, genre, url


def classify_track_tokens(
    *,
    div_id,
    data_id,
    data_isid,
    data_isided,
    data_mashpos,
    data_mashup,
    data_noguestsug,
    data_protected,
    data_rbcst,
    data_subpos,
    data_trackid,
    data_trno,
    data_videos,
    index,
    time,
    name,
    by_artist,
    publisher,
    duration,
    genre,
    url,
    track_IDer,
    total_tracklist_plays,
    is_id_remix,
) -> TrackToken:
    token_types: set[TrackTokenType] = set()

    if _is_true(data_mashup):
        token_types.add(TrackTokenType.MASHUP_TRACK)

    if _is_true(data_mashpos):
        token_types.add(TrackTokenType.MASHUP_CONSTITUENT_TRACK)

    if bool(is_id_remix):  # select_one(...) => Tag | None
        token_types.add(TrackTokenType.ID_REMIX_TRACK)

    if _is_true(data_isid):
        token_types.add(TrackTokenType.ID_TRACK)

    if not token_types:
        token_types.add(TrackTokenType.REGULAR_TRACK)

    return TrackToken(
        token_types=frozenset(token_types),
        attrs=TrackDOMAttrs(
            div_id=div_id,
            data_id=data_id,
            data_isid=data_isid,
            data_isided=data_isided,
            data_mashpos=data_mashpos,
            data_mashup=data_mashup,
            data_noguestsug=data_noguestsug,
            data_protected=data_protected,
            data_rbcst=data_rbcst,
            data_subpos=data_subpos,
            data_trackid=data_trackid,
            data_trno=data_trno,
            data_videos=data_videos,
        ),
        positional=PositionalInformation(index=index, time=time),
        metadata=TrackMetadata(
            name=name,
            by_artist=by_artist,
            publisher=publisher,
            duration_seconds=duration,
            genre=genre,
            url=url,
        ),
        reliability=ReliabilityMetrics(
            track_IDer=track_IDer,
            total_tracklist_plays=total_tracklist_plays,
            is_id_remix=bool(is_id_remix),
        ),
    )


def _tag_to_text(x):
    if isinstance(x, Tag):
        return x.get("content") or x.get_text(strip=True) or None
    return x

def tokenize_track_div(outer_div: Tag) -> TrackToken:
    div_id = outer_div.get("id")
    data_id = outer_div.get("data-id")
    data_isid = outer_div.get("data-isid")
    data_isided = outer_div.get("data-isided")
    data_mashpos = outer_div.get("data-mashpos")
    data_mashup = outer_div.get("data-mashup")
    data_noguestsug = outer_div.get("data-noguestsug")
    data_protected = outer_div.get("data-protected")
    data_rbcst = outer_div.get("data-rbcst")
    data_subpos = outer_div.get("data-subpos")
    data_trackid = outer_div.get("data-trackid")
    data_trno = outer_div.get("data-trno")
    data_videos = outer_div.get("data-videos")

    index, time = get_positional_information(outer_div)
    name, by_artist, publisher, duration, genre, url = get_metadata(outer_div)

    # In case get_metadata returns Tag objects
    name = _tag_to_text(name)
    by_artist = _tag_to_text(by_artist)
    publisher = _tag_to_text(publisher)
    genre = _tag_to_text(genre)
    url = _tag_to_text(url)

    ider_tag = outer_div.find("span", title="IDer of this track")
    plays_tag = outer_div.find("span", title="total tracklist plays")
    track_IDer = ider_tag.get_text(strip=True) if ider_tag else None
    total_tracklist_plays = plays_tag.get_text(strip=True) if plays_tag else None

    is_id_remix = outer_div.select_one("span.trackStatus")

    assert not (_is_true(data_mashup) and _is_true(data_mashpos)), "data_mashup and data_mashpos are orthogonal"
    assert not (_is_true(data_isid) and _is_true(data_isided)), "data_isid and data_isided are orthogonal"

    return classify_track_tokens(
        div_id=div_id,
        data_id=data_id,
        data_isid=data_isid,
        data_isided=data_isided,
        data_mashpos=data_mashpos,
        data_mashup=data_mashup,
        data_noguestsug=data_noguestsug,
        data_protected=data_protected,
        data_rbcst=data_rbcst,
        data_subpos=data_subpos,
        data_trackid=data_trackid,
        data_trno=data_trno,
        data_videos=data_videos,
        index=index,
        time=time,
        name=name,
        by_artist=by_artist,
        publisher=publisher,
        duration=duration,
        genre=genre,
        url=url,
        track_IDer=track_IDer,
        total_tracklist_plays=total_tracklist_plays,
        is_id_remix=is_id_remix,
    )


In [14]:
from dataclasses import dataclass
from enum import StrEnum
from bs4 import Tag


class SuggestionTokenType(StrEnum):
    TRACK_SUGGESTION = "track_suggestion"                      # 0
    WRONG_TRACK = "wrong_track"                                # 1
    TRACK_BEFORE = "track_before"                              # 2
    TRACK_AFTER = "track_after"                                # 3
    WITH_TRACK = "with_track"                                  # 4
    TRACK_NOT_PLAYED = "track_not_played"                      # 5
    TRACK_MISSPELLED = "track_misspelled"                      # 6
    ADDITIONAL_VIDEO = "additional_video"                      # 7
    TRACKNUMBER_SHOULD_BE = "tracknumber_should_be"            # 8
    PART_OF_MASHUP = "part_of_mashup"                          # 9
    PLAYED_WITH_PREVIOUS = "played_with_previous"              # 10
    CORRECT_LABEL = "correct_label"                            # 12
    CORRECT_HEADLINE = "correct_headline"                      # 13
    CORRECT_CUE_TIME = "correct_cue_time"                      # 14
    MASHUP_PART = "mashup_part"                                # 15
    TRACK_REWORK_OF = "track_rework_of"                        # 17
    FEATURE_ARTIST_CORRECTION = "feature_artist_correction"    # 18
    OTHER_CORRECTION = "other_correction"                      # 255
    USER_MARKED_INCORRECT = "user_marked_incorrect"            # None


TYPE_MAP: dict[str | None, SuggestionTokenType] = {
    "0": SuggestionTokenType.TRACK_SUGGESTION,
    "1": SuggestionTokenType.WRONG_TRACK,
    "2": SuggestionTokenType.TRACK_BEFORE,
    "3": SuggestionTokenType.TRACK_AFTER,
    "4": SuggestionTokenType.WITH_TRACK,
    "5": SuggestionTokenType.TRACK_NOT_PLAYED,
    "6": SuggestionTokenType.TRACK_MISSPELLED,
    "7": SuggestionTokenType.ADDITIONAL_VIDEO,
    "8": SuggestionTokenType.TRACKNUMBER_SHOULD_BE,
    "9": SuggestionTokenType.PART_OF_MASHUP,
    "10": SuggestionTokenType.PLAYED_WITH_PREVIOUS,
    "12": SuggestionTokenType.CORRECT_LABEL,
    "13": SuggestionTokenType.CORRECT_HEADLINE,
    "14": SuggestionTokenType.CORRECT_CUE_TIME,
    "15": SuggestionTokenType.MASHUP_PART,
    "17": SuggestionTokenType.TRACK_REWORK_OF,
    "18": SuggestionTokenType.FEATURE_ARTIST_CORRECTION,
    "255": SuggestionTokenType.OTHER_CORRECTION,
    None: SuggestionTokenType.USER_MARKED_INCORRECT,
}


@dataclass(frozen=True, slots=True)
class SuggestionToken:
    token_type: SuggestionTokenType
    div_id: str | None
    data_type: str | None
    data_guest: str | None
    data_nospam: str | None
    data_pos: str | None
    data_tlp: str | None
    data_track: str | None
    data_user: str | None
    data_value: str | None
    onclick: str | None
    cue: str | None
    value_text: str | None
    display_text: str | None


def tokenize_suggestion_div(outer_div: Tag) -> SuggestionToken:
    # Base attrs
    div_id = outer_div.get("id")
    data_type = outer_div.get("data-type")
    data_guest = outer_div.get("data-guest")
    data_nospam = outer_div.get("data-nospam")
    data_pos = outer_div.get("data-pos")
    data_tlp = outer_div.get("data-tlp")
    data_track = outer_div.get("data-track")
    data_user = outer_div.get("data-user")
    data_value = outer_div.get("data-value")
    onclick = outer_div.get("onclick")

    # Common extracted text fields
    cue_tag = outer_div.select_one("span.cue")
    cue = cue_tag.get_text(strip=True) if cue_tag else None

    value_tag = outer_div.select_one("div[id$='_value']")
    value_text = value_tag.get_text(" ", strip=True) if value_tag else None

    # Human-readable label like "track misspelled", "correct cue time is", etc.
    display_tag = outer_div.select_one("div.bCont > span.small")
    display_text = display_tag.get_text(" ", strip=True) if display_tag else None

    token_type = TYPE_MAP.get(data_type, SuggestionTokenType.UNKNOWN)

    return SuggestionToken(
        token_type=token_type,
        div_id=div_id,
        data_type=data_type,
        data_guest=data_guest,
        data_nospam=data_nospam,
        data_pos=data_pos,
        data_tlp=data_tlp,
        data_track=data_track,
        data_user=data_user,
        data_value=data_value,
        onclick=onclick,
        cue=cue,
        value_text=value_text,
        display_text=display_text,
    )


In [15]:
import re

tokens: TrackToken = []

def tokenize_row(row_raw_html: str):
    row_soup = BeautifulSoup(row_raw_html, 'html.parser')
    # Track row
    if (outer_div := row_soup.find("div", class_="tlpItem")):
        tokens.append(tokenize_suggestion_div(outer_div))
    # Suggestion Row
    elif (outer_div := row_soup.find("div", class_="sugTog")):
        tokens.append(tokenize_suggestion_div(outer_div))
        
    # Text Row (multiple different meanings)
    elif (outer_div := row_soup.find("div", class_="bItmH")):
        pass
    else:
        return


In [16]:
import sqlite3
from collections.abc import Iterator

from pathlib import Path
from typing import Iterator
import sqlite3

LIMIT = 50000

def iter_rows(db_path: Path, limit: int = 0, batch_size: int = 2000) -> Iterator[sqlite3.Row]:
    q = """
    SELECT
        r.set_id,
        s.set_url,
        r.element_id,
        r.classes,
        r.data_attrs_json,
        r.text_excerpt,
        r.raw_html
    FROM dj_set_rows r
    LEFT JOIN dj_sets s ON s.set_id = r.set_id
    """
    params: tuple[int, ...] = ()
    if limit > 0:
        q += " LIMIT ?"
        params = (int(limit),)

    with sqlite3.connect(str(db_path)) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.cursor()
        cur.execute(q, params)

        while True:
            batch = cur.fetchmany(batch_size)
            if not batch:
                break
            yield from batch

In [17]:
from bs4 import BeautifulSoup

def _is_true(v) -> bool:
    if v is None:
        return False
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() not in {"", "0", "false", "none", "null"}

violations = []

for i, row in enumerate(iter_rows(DB_PATH, LIMIT), 1):
    row_soup = BeautifulSoup(row["raw_html"], "html.parser")
    if not (outer_div := row_soup.find("div", class_="tlpItem")):
        continue

    track_attrs |= set(outer_div.attrs.keys())

    data_mashpos = outer_div.get("data-mashpos")
    data_mashup = outer_div.get("data-mashup")
    data_isid = outer_div.get("data-isid")
    data_isided = outer_div.get("data-isided")
    is_id_remix = outer_div.select_one("span.trackStatus")

    # existing orthogonality check
    assert not (_is_true(data_isid) and _is_true(data_isided)), \
        "data_isid and data_isided are orthogonal"

    # 4-way orthogonality check
    flags = {
        "mashup_track": _is_true(data_mashup),
        "mashup_constituent_track": _is_true(data_mashpos),
        "id_remix_track": bool(is_id_remix),
        "id_track": _is_true(data_isid),
    }
    active = [k for k, v in flags.items() if v]

    if len(active) > 1:
        violations.append((i, outer_div.get("id"), active))
        continue

    if i > LIMIT:
        break
    
# fail once, with useful sample
assert not violations, f"Non-orthogonal rows found: {len(violations)}; first 10={violations[:10]}"


NameError: name 'track_attrs' is not defined

In [18]:
data_type_examples = {}
id_examples = []
track_examples = []
mashup_examples = []
mashpos_examples = []
text_row_examples = []
track_attrs = set()
sug_attrs = set()
text_attrs = set()
# Efficient loop
for i, row in enumerate(iter_rows(DB_PATH, LIMIT), 1):
    # process row
    row_soup = BeautifulSoup(row["raw_html"], 'html.parser')
    if (outer_div := row_soup.find("div", class_="tlpItem")):
        track_attrs |= set(outer_div.attrs.keys())
        data_mashpos = outer_div.get("data-mashpos")
        data_mashup = outer_div.get("data-mashup")
        data_isid = outer_div.get("data-isid")
        data_isided = outer_div.get("data-isided")
        assert not (data_mashup and data_mashpos), "Data mashup and data pos are orthogonal!"
        assert not (data_isid and data_isided), "Data isid and data isided are orthogonal!"

        if data_isid:
            id_examples.append(row_soup)
        if data_isided and not data_mashup and not data_mashpos:
            track_examples.append(row_soup)
        if data_mashup:
            mashup_examples.append(row_soup)
        if data_mashpos:
            mashpos_examples.append(row_soup)
    # Suggestion Row
    elif (outer_div := row_soup.find("div", class_="sugTog")):
        continue
        # sug_attrs |= set(outer_div.attrs.keys())

        # data_type = outer_div.get("data-type")

        # if data_type not in data_type_examples:
        #     data_type_examples[data_type] = [outer_div]
        # else:
        #     data_type_examples[data_type].append(outer_div)
    # Text Row (multiple different meanings)
    elif (outer_div := row_soup.find("div", class_="bItmH")):
        text_row_examples.append(row_soup)
    else:
        continue
        # outer_div = row_soup.find("div")
        # is_player_widget = bool(outer_div and outer_div.get("id") == "playerWidget")
        # is_tl_save = bool(outer_div and outer_div.get("id") == "tl_save")

        # if is_player_widget or is_tl_save:
        #     continue

        # print(row["set_url"])
        # print(row_soup)
        # break

    if i > LIMIT:
        break


In [ ]:
data_type_examples

{'4': [<div class="bItm con sugTog ntB tlp_3500415" data-guest="26791744" data-nospam="true" data-pos="29" data-tlp="3500415" data-track="true" data-type="4" data-value="true" id="sug_407767" onclick="rowToggle('sug', event);"><div class="bPlay"><i class="fa-24 fa fa-plus-circle greenTxt" title="w/ suggestion"></i><span class="cue noWrap" id="sug407767_cue_value">40:00</span></div><div class="bCont tl"><span class="small spR">w/</span><div class="fontL" id="sug407767_value"><span class="blueTxt">WILL K &amp; Jebu - Boomshaka<a class="notranslate" href="/track/2rxx9slp/will-k-jebu-boomshaka/index.html" title="open track page" translate="no"><i class="fa fa-external-link fa-lg linkIcon mA tgHid"></i></a></span><span class="trackLabel noWrap blueTxt spL trackLabel" title="label">AXTONE<a class="notranslate tgHid" href="/label/3mdrzh/axtone-records/index.html" title="open label page" translate="no"><i class="fa fa-external-link fa-lg linkIcon mA"></i></a></span></div><div class="wRow hOO">

In [21]:
text_row_examples

[<div class="bItmH"> <span class="fRow"> <a class="notranslate" href="/dj/alesso/index.html">Alesso</a> <span class="spL">played:</span> <i class="fa fa-24 fa-trash redTxt tlEdit spL action" id="0artistTracksDel" onclick="new MessageBox(this, { type: 'confirm', cancelButton: true, submitBind: function() { submitForm($('#0artistTracksDel'), { anker: 'tlp_1347425', artist1714_tracks_delete: true }); } }).show('Sure to delete all tracks played by Alesso?');" title="delete all tracks played by Alesso"> </i> </span> </div>,
 <div class="bItmH"> <span class="fRow"> <a class="notranslate" href="/dj/martingarrix/index.html">Martin Garrix</a> <span class="spL spR">&amp;</span><a class="notranslate" href="/dj/alesso/index.html">Alesso</a> <span class="spL">played:</span> <i class="fa fa-24 fa-trash redTxt tlEdit spL action" id="2artistTracksDel" onclick="new MessageBox(this, { type: 'confirm', cancelButton: true, submitBind: function() { submitForm($('#2artistTracksDel'), { anker: 'tlp_1347438',